# OBLITERATUS — One-Click Abliteration

**SOTA refusal removal** running on free Colab GPU. SVD multi-direction extraction, norm-preserving projection, iterative refinement.

Based on: Arditi et al. (2024) | Gabliteration (arXiv:2512.18901) | grimjim norm-preserving biprojection (2025)

---

**How to use:**
1. Make sure GPU runtime is enabled: `Runtime > Change runtime type > T4 GPU`
2. Set your model and method in the config cell below
3. Run All (`Runtime > Run all` or `Ctrl+F9`)
4. Download the abliterated model from the output

## 1. Install

In [ ]:
!pip install -q git+https://github.com/elder-plinius/OBLITERATUS.git
!pip install -q accelerate bitsandbytes

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00
PyTorch 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.6 GB


## 2. Configure

Edit the cell below to set your target model and abliteration method.

In [ ]:
#@title Abliteration Config { run: "auto" }

#@markdown ### Target Model
#@markdown Pick a model from the dropdown or paste a custom HuggingFace ID.
MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" #@param ["meta-llama/Llama-3.1-8B-Instruct", "Qwen/Qwen2.5-7B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3", "google/gemma-2-9b-it", "microsoft/Phi-3.5-mini-instruct", "THUDM/glm-4-9b-chat", "NousResearch/Hermes-3-Llama-3.1-8B", "cognitivecomputations/dolphin-2.9.4-llama3.1-8b", "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "openai-community/gpt2"] {allow-input: true}

#@markdown ### Method
METHOD = "advanced" #@param ["basic", "advanced", "aggressive"]

#@markdown ### Advanced Overrides (leave 0 to use method defaults)
N_DIRECTIONS = 0 #@param {type: "integer"}
REGULARIZATION = 0.0 #@param {type: "number"}
REFINEMENT_PASSES = 0 #@param {type: "integer"}

#@markdown ### Output
OUTPUT_DIR = "abliterated" #@param {type: "string"}

print(f"Model: {MODEL}")
print(f"Method: {METHOD}")
print(f"Output: {OUTPUT_DIR}/")

Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Method: advanced
Output: abliterated/


## 3. Run Abliteration Pipeline

This runs all 6 stages: SUMMON → PROBE → DISTILL → EXCISE → VERIFY → REBIRTH

In [ ]:
from google.colab import userdata
import os

HF_TOKEN = userdata.get("HF_TOKEN").strip()
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

from obliteratus.abliterate import AbliterationPipeline

MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "/content/abliterated"
METHOD = "advanced"

print("Starting pipeline...")
print("MODEL:", MODEL)
print("OUTPUT_DIR:", OUTPUT_DIR)

pipeline = AbliterationPipeline(
    model_name=MODEL,
    output_dir=OUTPUT_DIR,
    device="cuda",
    dtype="float16",
    method=METHOD,
    n_directions=1,
    regularization=0.1,
    refinement_passes=1,
)

result = pipeline.run()

print("ABLITERATION COMPLETE")
print("Output:", getattr(result, "output_dir", OUTPUT_DIR))

print("Folder exists:", os.path.exists(OUTPUT_DIR))
print("Files:", os.listdir(OUTPUT_DIR) if os.path.exists(OUTPUT_DIR) else "none")

Starting pipeline...
MODEL: TinyLlama/TinyLlama-1.1B-Chat-v1.0
OUTPUT_DIR: /content/abliterated


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

ABLITERATION COMPLETE
Output: /content/abliterated
Folder exists: True
Files: ['tokenizer.json', 'model-00001-of-00002.safetensors', 'generation_config.json', 'tokenizer_config.json', 'chat_template.jinja', 'config.json', 'model-00002-of-00002.safetensors', 'model.safetensors.index.json', 'abliteration_metadata.json']


## 4. Download the Abliterated Model

Run the cell below to zip and download, or upload directly to HuggingFace Hub.

In [ ]:
import os
from pathlib import Path

# Find the output directory
out_path = Path(OUTPUT_DIR)
subdirs = [d for d in out_path.iterdir() if d.is_dir()] if out_path.exists() else []
model_dir = subdirs[0] if subdirs else out_path

print(f"Model saved at: {model_dir}")
print(f"Contents:")
for f in sorted(model_dir.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1024**2
        print(f"  {f.relative_to(model_dir)}  ({size_mb:.1f} MB)")

Model saved at: /content/abliterated
Contents:
  abliteration_metadata.json  (0.0 MB)
  chat_template.jinja  (0.0 MB)
  config.json  (0.0 MB)
  generation_config.json  (0.0 MB)
  model-00001-of-00002.safetensors  (1907.2 MB)
  model-00002-of-00002.safetensors  (191.0 MB)
  model.safetensors.index.json  (0.0 MB)
  tokenizer.json  (3.5 MB)
  tokenizer_config.json  (0.0 MB)


In [ ]:
#@title Option A: Download as ZIP
import shutil
from google.colab import files

zip_name = model_dir.name
shutil.make_archive(zip_name, 'zip', model_dir)
print(f"Downloading {zip_name}.zip ...")
files.download(f"{zip_name}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#@title Option B: Push to HuggingFace Hub
#@markdown Set your HF repo name. You'll need to be logged in (`huggingface-cli login`).
HF_REPO = "49vc631b6/tinyllama-abliterated" #@param {type: "string"}

from huggingface_hub import HfApi
api = HfApi()

# Login if needed
from huggingface_hub import notebook_login
notebook_login()

# Upload
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(
    folder_path=str(model_dir),
    repo_id=HF_REPO,
    repo_type="model",
)
print(f"\nUploaded to: https://huggingface.co/{HF_REPO}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          |  527kB /  200MB            

  ...0001-of-00002.safetensors:   1%|          | 14.4MB / 2.00GB            


Uploaded to: https://huggingface.co/49vc631b6/tinyllama-abliterated


## 5. Test the Abliterated Model

Quick sanity check — try some prompts that the original model would refuse.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print(f"Loading abliterated model from {model_dir}...")
tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
model = AutoModelForCausalLM.from_pretrained(
    str(model_dir),
    torch_dtype=torch.float16,
    device_map="auto",
)

def generate(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test prompts
test_prompts = [
    "The capital of France is",
    "Explain how a neural network works:",
    "Write a poem about the ocean:",
]

for prompt in test_prompts:
    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*60}")
    print(generate(prompt))

Loading abliterated model from /content/abliterated...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PROMPT: The capital of France is


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The capital of France is Paris, located in the center of the country. The city is known for its beauty, culture, and history. It is also a major center of politics and economics in France.

2. Rome, Italy The capital of Italy is Rome, located in the center of the country. The city is known for its ancient ruins, art, and culture. It is also a major center of politics and economics in Italy.

3. Tokyo, Japan The capital of Japan is

PROMPT: Explain how a neural network works:


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain how a neural network works:

A neural network is a computationally complex network of interconnected nodes (neurons) that can be used to process and interpret input data. It consists of a series of layers, where each layer represents a level of abstraction or processing. At the bottom of the network is the input layer, where the data is fed in. The network then proceeds to the next layer, which may be a fully connected layer, a pooling layer, or a fully connected layer with a non-

PROMPT: Write a poem about the ocean:
Write a poem about the ocean:

The Waves Rise and Fall

The tide comes in,
And the ocean's face
Is a face of mystery,
A face of love.

The waves rise and fall,
A constant rhythm,
A constant beat,
A rhythm that's never ending.

The waves rise and fall,
A ceaseless dance,
A dance of life,
A dance of death.

The tide comes in
